# What do the SARS-Cov-2 genes do?


One process in the analysis of a new organism is annotating the genome. The first step of this analysis is to find potential genes through identifying open reading frames (ORFs). 
After this, the next question is which of these correspond to actual genes?

To answer this, we will lean on what we know about evolution. If two species are morphologically different, their genetic material must also differ substantially. Right? Not exactly. Evolution conserved many genes crucial for survival. The degree of conservation is surprising. For example, while we would expect to share many genes with the chimpanzee, our closest relative, we would expect almost no similarity in our genetic material with bananas. The numbers tell us otherwise. While sharing 96% of our genes with the chimpanzee, we also share 60% of our genes with bananas! These similar genes in humans and bananas encode proteins that carry similar functions.

Two genes with similar sequences may perform similar functions. Genes like these are called homologous genes. Or, more precisely, in our case, these would be orthologous genes (see the figure below). A pair of orthologous genes are genes from two related species originating from a common ancestor (e.g., humans and chimpanzees). Due to the evolution, the two nucleotide or protein sequences might differ slightly but not overwhelmingly.

<div>
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d8/Ortholog_paralog_analog_%28homologs%29.svg/1200px-Ortholog_paralog_analog_%28homologs%29.svg.png" width="500">
</div>

*Author: Thomas Shafee, Licence: CC BY 4.0*

The human-banana example demonstrates that these two organisms don't have to be that closely related! If we tried to determine the gene function in humans using bananas as a reference, we could infer the role of over 60% of the genes. However, using two closely related organisms would undoubtedly give us more robust results. We would be better off choosing the genome of a mouse or a chimpanzee as our reference.

Scientists have studied hundreds of viruses and determined the functions of most of their genes. Using viruses similar in sequence to SARS-Cov-2, we can characterize the SARS-CoV-2 ORFs to determine if they are the actual genes, and think about their function. Knowing what each gene does might help us find a way to fight the infection and potentially even find treatments for COVID-19. 

#### Please ensure each answer is completed in the single cell of the notebook immediately following the question, and that all answers are in ```markdown``` format.

Once you have completed the notebook, make sure you save it and submit it via Moodle.

All questions are worth between 4 and 10 marks. There are a total of *40 marks* available in this workbook. There is an extra bonus problem for *10 marks*. 
  
Total notebook marks will be scaled to a **course mark out of 10** to contribute to your course total.

In [1]:
import numpy as np
import os
import csv
import json
from Bio import Entrez, SeqIO
from Bio.Align import substitution_matrices
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import helper_functions
from helper_functions import global_alignment, local_alignment

In [2]:
# In order to import from the python file without hassle, we add the current
# directory to the python path
import sys; sys.path.append(".")

Let's let the nice folks at NCBI know who we are.

In [3]:
Entrez.email = "z5708961@student.unsw.edu.au"
PROTEIN_GAP = -4
MOTIF_GAP = -10
CACHE_DIR = "genbank_cache"

## Finding SARS-CoV-2's closest relatives

In the first part of this assignment, we will analyze several coronavirus genomes to determine which are most similar to SARS-CoV-2. We will then use the viruses with the most similar sequence as reference genomes to assign functions to SARS-CoV-2 ORFs.

To find the closest reference genomes, we will align the SARS-CoV-2 sequence to each candidate coronavirus genome. The virus genomes with the highest alignment scores will be our most closely-related viruses.

Notice that computing an alignment between two sequences of length $N$ and $M$ requires the computation of a dynamic programming table with $N \cdot M$ entries. The size of this matrix is sufficiently small for short sequences. But even the short genomes, like viral genomes, are generally too long for this approach. Instead of computing similarities from the entire genomes, we will only focus on the spike protein sequence. In general, we would prefer to align the whole nucleotide sequences, but for this assignment, considering only a spike protein will be sufficient. The spike protein is also one of the essential parts of any coronavirus, as it is the one that grants the virus entry to host cells. 

Considering spike proteins alone reduces the sequence lengths from ~30k (entire virus) to around 1.3k (spike protein). Nevertheless, do your best to write fast, efficient Python code. Each `global_alignment` call on 1.3k long protein sequences should take around 30 to 60 seconds. 
We have to calculate 20 comparisons, which takes roughly 10 to 20 minutes.

In [4]:
accession_codes = {
    # 6 known human coronaviruses
    "Human-SARS": "NC_004718",
    "Human-MERS": "NC_019843",
    "Human-HCoV-OC43": "NC_006213",
    "Human-HCoV-229E": "NC_002645",
    "Human-HCoV-NL63": "NC_005831",
    "Human-HCoV-HKU1": "NC_006577",
    
    # Bat
    "Bat-CoV MOP1": "EU420138",
    "Bat-CoV HKU8": "NC_010438",
    "Bat-CoV HKU2": "NC_009988",
    "Bat-CoV HKU5": "NC_009020",
    "Bat-CoV RaTG13": "MN996532",
    "Bat-CoV-ENT": "NC_003045",
    
    # Other animals
    "Hedgehog-CoV 2012-174/GER/2012": "NC_039207",
    "Pangolin-CoV MP789": "MT121216",
    "Rabbit-CoV HKU14": "NC_017083",
    "Duck-CoV isolate DK/GD/27/2014": "NC_048214",
    "Feline infectious peritonitis virus": "NC_002306",  # cat
    "Giraffe-CoV US/OH3/2003": "EF424623",
    "Murine-CoV MHV/BHKR_lab/USA/icA59_L94P/2012": "KF268338",  # mouse
    "Equine-CoV Obihiro12-2": "LC061274",  # horse
}

Above is the list of coronaviruses and their corresponding NCBI accession codes that we will consider in this assignment. You can find the SARS-CoV-2 sequence in `data/sars_cov_2.fa`. This time, we will use the FASTA format to familiarize ourselves with the sequence file formats used in bioinformatics. You can easily read FASTA files using the `SeqIO.read` function from `biopython`.

We will only use part of the sequence, and to assess similarity, consider the spike protein alone. We will assume that we know the location of the spike protein in SARS-CoV-2 (strand=1, start=21562, stop=25384). We need to find a part of the sequence that encodes for the spike protein in all coronaviruses listed above. We can inspect each sequence's features to locate the spike protein. For this assignment, we will iterate through gene coding regions (CDS) for each coronavirus and find the one that codes for the "S" gene. Some coronaviruses do not include such annotation but instead report on a "spike protein" in the `product` field.

## Problem 1: Global alignment

**Task:** Use the template `global_alignment` function in the helper file to implement the Needleman-Wunsch algorithm for global sequence alignment. Use a "-" character for indels. Score the alignments using the BLOSUM62 substitution matrix. You can find it in `biopython`.

**[10 points]**

In [5]:
blosum62 = substitution_matrices.load("BLOSUM62")


def blosum_score(a, b):
    return blosum62[a, b]


def load_record(accession):
    if not os.path.exists(CACHE_DIR):
        os.makedirs(CACHE_DIR)
    path = os.path.join(CACHE_DIR, accession + ".gb")
    if not os.path.exists(path):
        print("downloading", accession)
        handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text")
        text = handle.read()
        handle.close()
        f = open(path, "w", encoding="utf-8")
        f.write(text)
        f.close()
    return SeqIO.read(path, "genbank")


def get_spike(record):
    for feature in record.features:
        if feature.type == "CDS" and "translation" in feature.qualifiers:
            gene = feature.qualifiers.get("gene", [""])[0]
            product = feature.qualifiers.get("product", [""])[0]
            if gene == "S" or "spike" in product.lower():
                return feature.qualifiers["translation"][0]
    return None


def percent_identical(a1, a2):
    if len(a1) == 0:
        return 0.0
    matches = 0
    for i in range(len(a1)):
        if a1[i] == a2[i]:
            matches = matches + 1
    return 100.0 * matches / len(a1)


genome = SeqIO.read("data/sars_cov_2.fa", "fasta")

## Problem 2: Choosing our reference genomes

**Task:** Identify the three closest relatives to SARS-CoV-2. Use the `global_alignment` function you have just implemented, inspect their alignment scores, and select the three most similar sequences to our spike protein sequence. These will serve as reference genomes in Problem 4 of this assignment and help us determine the function of ORFs.

Save the full names of three coronaviruses with the closest matches to SARS-CoV-2's spike protein sequence as a Python list in the `three_closest_references` variable.

**[6 points]**

Then answer the following questions about global alignment and the three closest references.
- When aligning spike protein sequences with the global alignment, we used the BLOSUM62 substitution matrix by default. Examine the similarity of the protein alignments (the frequency of matching amino acids) and discuss whether it would be better to use the BLOSUM90 or BLOSUM45 substitution matrix in our case.
- The theory of evolution states that the SARS-CoV-2 virus must have evolved from some other virus in the past. Based on the three closest alignment references, is it more likely that SARS-CoV-2 jumped from another host organism or evolved from a human coronavirus? (We will explore this further in the following exercise.)

Store your answers in the `blosum_selection` and `viral_origin` variables, respectively.

**[4 points]**.

In [6]:
helper_functions.GAP_PENALTY = PROTEIN_GAP

spike2 = str(genome.seq[21562:25384].translate()).replace("*", "")

scores = {}
identities = {}
spikes = {}

for name in accession_codes:
    record = load_record(accession_codes[name])
    spike = get_spike(record)
    if spike is None:
        print("no spike found for", name)
        continue
    spikes[name] = spike
    a1, a2, score = global_alignment(spike2, spike, blosum_score)
    scores[name] = score
    identities[name] = percent_identical(a1, a2)
    print(name, score, round(identities[name], 1))

three = []


Human-SARS 5246.0 76.4
Human-MERS 1739.0 34.4
Human-HCoV-OC43 1617.0 32.7
Human-HCoV-229E 1101.0 29.9
Human-HCoV-NL63 1158.0 29.7
Human-HCoV-HKU1 1544.0 32.0
Bat-CoV MOP1 1087.0 29.4
Bat-CoV HKU8 1153.0 29.4
Bat-CoV HKU2 1091.0 28.7
Bat-CoV HKU5 1797.0 35.1
Bat-CoV RaTG13 6539.0 97.4
Bat-CoV-ENT 1629.0 32.5
Hedgehog-CoV 2012-174/GER/2012 1694.0 33.1
Pangolin-CoV MP789 6085.0 90.1
Rabbit-CoV HKU14 1653.0 32.6
Duck-CoV isolate DK/GD/27/2014 1124.0 30.8
Feline infectious peritonitis virus 1150.0 29.2
Giraffe-CoV US/OH3/2003 1623.0 31.8
Murine-CoV MHV/BHKR_lab/USA/icA59_L94P/2012 1719.0 34.1
Equine-CoV Obihiro12-2 1575.0 32.0


In [7]:
for k in range(3):
    best_name = None
    for name in scores:
        if name in three:
            continue
        if best_name is None or scores[name] > scores[best_name]:
            best_name = name
    three.append(best_name)
print("three closest:", three)

three closest: ['Bat-CoV RaTG13', 'Pangolin-CoV MP789', 'Human-SARS']


In [ ]:
blosum_selection = "BLOSUM90 is a suitable choice because the three closest spike proteins have high sequence identity to SARS-CoV-2: Bat-CoV RaTG13 (97.4%), Pangolin-CoV MP789 (90.1%), and Human-SARS (76.4%). BLOSUM90 is designed for closely related sequences, while BLOSUM45 is intended for more distant sequences."


In [ ]:
viral_origin = "The three closest references are Bat-CoV RaTG13, Pangolin-CoV MP789, and Human-SARS. Two of these three references are animal coronaviruses, so SARS-CoV-2 is more similar to animal coronaviruses than to most human coronaviruses in this dataset. However, sequence similarity alone cannot establish the exact host or route of transmission."

## MiniBLAST

A sophisticated approach for characterizing any genetic sequence is BLAST. BLAST stores a database of annotated genes (usually from various organisms) and then uses local alignment and heuristic search to find database genes with a similar sequence to a query one. Assuming that genes with similar protein sequences perform similar functions, we can infer the function of a query sequence by looking at the function of any matching genes returned by BLAST. It is also possible that BLAST will not find suitable matches, which may signal that a sequence might not be an actual gene or protein. In this problem, we will implement our simplified BLAST version and call it MiniBLAST. We will use MiniBLAST to determine whether each of our candidate ORFs is a gene. And if it is a gene, we will use the annotations to determine what the gene does.

Please note that BLAST is a complicated, optimized, heuristic, state-of-the-art technique to obtain very good approximate solutions. BLAST can query thousands of sequences in seconds. Our implementation will be slightly less sophisticated and somewhat more constrained but conceptually similar to that of BLAST.

## Problem 3: Local alignment

**Task:** Use the template `local_alignment` in the helper file and implement the Smith-Waterman algorithm for local sequence alignment.

**[10 points]**

## Problem 4: Finding homologous genes

For this assignment, we have selected five promising SARS-CoV-2 ORFs. 

In [10]:
orf_candidates = {
    "ORF-1": (1, 11995, 13483),
    "ORF-2": (1, 26792, 27191),
    "ORF-3": (1, 23650, 25384),
    "ORF-4": (-1, 421, 667),
    "ORF-5": (1, 9133, 13483),
}

We will now use MiniBLAST to compare these ORFs to known, annotated genes from the three closely-related coronaviruses you have stored in `three_closest_references`. The goal is to find the best matching gene from the reference genomes and decide if the quality of the match is sufficient to claim that ORF is an actual gene. 

Hint: among the five ORFs included in the candidates, we have planted one that is not a true gene.

**Task:** We start by building a reference database. For each of the three closely-related coronaviruses from Problem 2, extract the coding regions (CDS) from each genome, and convert them to protein sequences. Store the name of the virus each protein sequence came from and its function. In our case, the gene's function is evident from its name, e.g., "spike protein", "ORF1ab", and similar. Given `orf_candidates`, compute the local alignment to each protein sequence in our database. Inspect each alignment and its corresponding score, and decide whether the alignment is sufficiently good to assume they perform the same function.

Save your answers into the `orf_matches` variable as indicated in the cell below. Each ORF should be assigned a *closest-organism*, reporting the reference virus with the closest match, and a *homologous-gene*, indicating which gene the ORF matched to. If two ORFs match the reference with the same score, report one of them. For ORFs with no match in the reference, use None for both fields.

**[10 points]**

In [11]:
database = []
for virus in three:
    record = load_record(accession_codes[virus])
    for feature in record.features:
        if feature.type == "CDS" and "translation" in feature.qualifiers:
            product = feature.qualifiers.get("product", ["unknown"])[0]
            database.append((virus, product, feature.qualifiers["translation"][0]))
print(len(database), "proteins in database")

orf_matches = {}
for orf in orf_candidates:
    strand = orf_candidates[orf][0]
    start = orf_candidates[orf][1]
    stop = orf_candidates[orf][2]
    dna = genome.seq[start:stop]
    if strand == -1:
        dna = dna.reverse_complement()
    protein = str(dna.translate()).replace("*", "")

    self_score = 0
    for letter in protein:
        self_score = self_score + blosum62[letter, letter]

    best_score = 0
    best_virus = None
    best_gene = None
    for item in database:
        a1, a2, score = local_alignment(protein, item[2], blosum_score)
        if score > best_score:
            best_score = score
            best_virus = item[0]
            best_gene = item[1]

    fraction = 0
    if self_score > 0:
        fraction = best_score / self_score
    print(orf, len(protein), best_score, round(fraction, 2), best_virus, best_gene)

    if fraction >= 0.3:
        orf_matches[orf] = {"closest-organism": best_virus, "homologous-gene": best_gene}
    else:
        orf_matches[orf] = {"closest-organism": None, "homologous-gene": None}


36 proteins in database
ORF-1 495 2553.0 0.99 Bat-CoV RaTG13 orf1ab polyprotein
ORF-2 132 678.0 1.0 Bat-CoV RaTG13 membrane protein
ORF-3 577 2999.0 1.0 Bat-CoV RaTG13 spike glycoprotein
ORF-4 81 61.0 0.14 Human-SARS ORF1ab polyprotein
ORF-5 1449 7569.0 0.99 Bat-CoV RaTG13 orf1ab polyprotein


## Bonus Problem: Repurposing SARS-CoV drug treatments for SARS-CoV-2

You might have noticed that the coronavirus we've been working with ends with the number 2. That means there must have been a SARS-CoV-1 at some point, right? Indeed there was. The SARS-CoV-1 virus caused a SARS outbreak in 2002-2004 and infected over 8,000 people from 29 countries, killing at least 774 people. These may seem like small numbers today, but at the time, the outbreak caused much panic because the virus had a much higher mortality rate than SARS-CoV-2. 

Several drugs were developed and tested for the treatment of SARS. Some proved effective. Since SARS-CoV-2 is closely related to SARS-CoV-1, we could repurpose some effective drugs for SARS-CoV-1. We will look at one drug in particular: EK1 binds to a specific region on the SARS-CoV-1 spike protein and prevents the virus from entering the cells, thus stopping the infection and, consequently, the spread of the virus. Assuming the spike protein in SARS-CoV-2 is similar enough to the spike protein in SARS-CoV-1, EK1 may be able to bind to the SARS-CoV-2 spike protein and subsequently stop the infection.

EK1 recognizes a specific sequence (or motif) of aminoacid types: **_hpphcpc_** where **_h_** is a hydrophobic, **_p_** is a polar, and **_c_** is a charged amino acid. There are approximately six consecutive repeats of this motif in SARS-CoV-1, so we expect to find a similar pattern in the SARS-CoV-2 spike protein. The individual amino acid types in this motif are not equally important for the successful binding of EK1. For instance, it is much more important for the **_h_** amino acids to be in the correct spots than the **_p_** or **_c_** amino acids. It is also critical that none of the amino acids are missing in the spike protein, as this will prevent binding.

Aminoacids and their corresponding types (**_h_**, **_p_**, or **_c_**) are provided in `data/aminoacid_properties.csv`.

**Task:** Determine whether EK1 can bind to the SARS-CoV-2 spike protein. Based on the description of how EK1 binds to the spike protein in the paragraphs above, design a scoring function that will appropriately weigh amino acid matches/mismatches. Then, perform local alignment between the SARS-CoV-1 spike protein sequence and the EK1 target motif (**_hpphcpc_** repeated six times). Since EK1 was developed for SARS-CoV-1, we know there should be a valid binding motif somewhere along the SARS-CoV-1 spike protein. Use this fact to calibrate your scoring function. Can we find an EK1 binding motif in the SARS-CoV-2 spike protein? Plot the positions of the best alignments in both viruses. For each virus, plot a horizontal line representing the spike protein sequence and a bar indicating the location of the EK1 binding motif. Ensure the two genomes are aligned to see whether the binding motifs appear in the same general regions of the spike protein. Save your figure into `problem-bonus.png`.

**[5 points]**

Then answer the following questions:
- Does the SARS-CoV-2 spike protein contain the EK1 binding motif?
- Are SARS-CoV-1 and SARS-CoV-2 motifs comparable? What procentage of the aminoacids match between the motifs?
- Since the introduction of EK1 as a potential COVID-19 treatment, the drug has seen further development. Does the treatment work on COVID-19 patients? Search through literature and provide at least two references.

Store your answers in the `binding_motif_present`, `binding_motif_similarity` and `EK1_drug_efficacy` variables, respectively.

**[5 points]**

*Hint*: Our task is to design a scoring function that reflects the underlying process of binding EK1 to the spike protein. For instance, since no amino acids must be missing from the spike protein sequence, we can design our scoring function to disallow any indels in our alignment. We can achieve this by heavily penalizing indels. Similarly, if we know hydrophobic matches are more important than polar or charged matches, we can appropriately weigh matches and penalize mismatches between these types of proteins.



In [12]:
types = {}
f = open("data/aminoacid_properties.csv", newline="", encoding="utf-8")
for row in csv.DictReader(f):
    types[row["aminoacid"]] = row["polarity"]
f.close()


def ek1_score(amino_acid, motif_letter):
    if types.get(amino_acid) == motif_letter:
        if motif_letter == "h":
            return 4
        return 2
    if motif_letter == "h":
        return -4
    return -1


motif = "hpphcpc" * 6
spike1 = spikes["Human-SARS"]

helper_functions.GAP_PENALTY = MOTIF_GAP
hit1, m1, score1 = local_alignment(spike1, motif, ek1_score)
hit2, m2, score2 = local_alignment(spike2, motif, ek1_score)
helper_functions.GAP_PENALTY = PROTEIN_GAP

hit1 = hit1.replace("-", "")
hit2 = hit2.replace("-", "")
start1 = spike1.find(hit1)
start2 = spike2.find(hit2)
print("SARS-CoV-1 hit", start1 + 1, start1 + len(hit1), score1, hit1)
print("SARS-CoV-2 hit", start2 + 1, start2 + len(hit2), score2, hit2)

b1, b2, s = global_alignment(hit1, hit2, blosum_score)
motif_identity = percent_identical(b1, b2)
print("motif identity", round(motif_identity, 1))

plt.figure(figsize=(10, 3))
plt.hlines(1, 0, len(spike1), color="black")
plt.hlines(0, 0, len(spike2), color="black")
plt.barh(1, len(hit1), left=start1, height=0.3, color="red")
plt.barh(0, len(hit2), left=start2, height=0.3, color="red")
plt.yticks([0, 1], ["SARS-CoV-2 spike", "SARS-CoV-1 spike"])
plt.ylim(-0.6, 1.6)
plt.xlabel("position in spike protein (amino acids)")
plt.title("EK1 binding motif (red bar) location")
plt.tight_layout()
plt.savefig("problem-bonus.png", dpi=150)
plt.close()




SARS-CoV-1 hit 938 974 51.0 AQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQ
SARS-CoV-2 hit 956 992 51.0 AQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQ
motif identity 100.0


In [ ]:
binding_motif_present = "The recorded local alignment identifies a candidate EK1 binding motif in the SARS-CoV-2 spike protein at positions 956-992, with an alignment score of 51. The SARS-CoV-1 hit is at positions 938-974, also with a score of 51. These results support the presence of a comparable motif in SARS-CoV-2."

binding_motif_similarity = "The aligned SARS-CoV-1 and SARS-CoV-2 motif regions have 100.0% identical amino acids in the recorded alignment. The regions occur in a similar part of the spike proteins, as shown in problem-bonus.png."

EK1_drug_efficacy = "Studies report that EK1 and its derivatives have antiviral activity in laboratory experiments and animal models. Xia et al. (Cell Research, 2020) reported that EK1C4, a cholesterol-linked derivative of EK1, inhibited SARS-CoV-2 spike-mediated fusion and pseudovirus infection, and reported protective effects in mice. Further research has reported EK1 activity against variants in cell and animal experiments. These preclinical findings do not establish that EK1 is an effective treatment for COVID-19 patients, and the cited studies do not provide evidence of clinical efficacy in humans."